# NestedPandasDataset Example

Minimal example using `NestedPandasDataset` with a [nested-pandas](https://nested-pandas.readthedocs.io/) parquet file. The dataset exposes one `get_<column>` method per top-level column. Nested columns return a `dict` of NumPy arrays for the requested object.

In [ ]:
from pathlib import Path
import tempfile

import pandas as pd
import nested_pandas

import hyrax

In [ ]:
tmp_dir = Path(tempfile.mkdtemp())
parquet_path = tmp_dir / "sample.parquet"

base = pd.DataFrame(
    {
        "object_id": [1001, 1002, 1003],
        "ra": [150.1, 150.2, 150.3],
    },
    index=[0, 1, 2],
)

lightcurve = pd.DataFrame(
    {
        "mjd": [59000.0, 59001.0, 59002.0, 59003.0, 59004.0, 59005.0],
        "flux": [10.0, 11.0, 20.0, 21.0, 30.0, 31.0],
    },
    index=[0, 0, 1, 1, 2, 2],
)

nf = nested_pandas.NestedFrame(base).join_nested(lightcurve, "lightcurve")
nf.to_parquet(parquet_path)
parquet_path

In [ ]:
h = hyrax.Hyrax()
h.config["data_request"] = {
    "train": {
        "data": {
            "dataset_class": "NestedPandasDataset",
            "data_location": str(parquet_path),
            "fields": ["ra", "lightcurve"],
            "primary_id_field": "object_id",
        }
    }
}
dataset = h.prepare()["train"]
len(dataset)

In [ ]:
dataset[0]